In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/cleaned/playstore_features.csv")

feature_cols_num = ["Size_MB", "Price", "Days_Since_Update"]
feature_cols_cat = ["Category", "Content Rating"]
X = df[feature_cols_num + feature_cols_cat].copy()
y = df["Success"].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), feature_cols_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_cat),
])

rf = Pipeline([
    ("pre", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=20,
                                     class_weight="balanced", random_state=42, n_jobs=-1))
])
rf.fit(X_train, y_train)
print("Model retrained. Test ROC-AUC:", round(__import__("sklearn.metrics", fromlist=["roc_auc_score"]).roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]), 3))

Model retrained. Test ROC-AUC: 0.828


In [2]:
# From Step 4 (SQL): success rate + top-10 concentration
cat_stats = (df.groupby("Category")
             .filter(lambda x: len(x) >= 50)
             .groupby("Category")
             .agg(n_apps=("Success", "count"), success_rate=("Success", "mean"), avg_rating=("Rating", "mean")))
cat_stats["success_rate"] *= 100

def top10_share(group):
    total = group["Installs"].sum()
    top10 = group.nlargest(10, "Installs")["Installs"].sum()
    return 100 * top10 / total

conc = df.groupby("Category").filter(lambda x: len(x) >= 50).groupby("Category").apply(top10_share)
cat_stats["top10_concentration"] = conc

# Opportunity score: reward high success, penalize high concentration
cat_stats["opportunity_score"] = cat_stats["success_rate"] - 0.5 * cat_stats["top10_concentration"]

decision_table = cat_stats.sort_values("opportunity_score", ascending=False).round(1)
decision_table.head(10)

,n_apps,success_rate,avg_rating,top10_concentration,opportunity_score
Category,,,,,
ENTERTAINMENT,86,83.7,4.1,72.1,47.7
GAME,945,57.0,4.2,26.0,44.0
PHOTOGRAPHY,281,61.6,4.2,40.8,41.2
EDUCATION,105,58.1,4.3,47.6,34.3
SHOPPING,202,52.5,4.2,53.6,25.7
FOOD_AND_DRINK,112,43.8,4.2,47.2,20.1
WEATHER,79,53.2,4.2,72.0,17.2
FAMILY,1877,30.3,4.2,30.5,15.0
VIDEO_PLAYERS,164,51.8,4.0,80.1,11.8


In [3]:
# Pick your top recommended category from decision_table.head(1)
TARGET_CATEGORY = decision_table.index[0]
print("Recommended category:", TARGET_CATEGORY)

# "Typical" app = median values across the whole dataset
baseline_scenario = X_test.copy()

# "Recommended" app = target category + best practices from Step 7's top drivers
recommended_scenario = X_test.copy()
recommended_scenario["Category"] = TARGET_CATEGORY
recommended_scenario["Size_MB"] = df["Size_MB"].quantile(0.75)      # larger apps succeed more (Step 6/7)
recommended_scenario["Days_Since_Update"] = 15                       # frequent updates
recommended_scenario["Price"] = 0                                    # free

baseline_prob = rf.predict_proba(baseline_scenario)[:, 1].mean()
recommended_prob = rf.predict_proba(recommended_scenario)[:, 1].mean()

print(f"Baseline (typical app) predicted success probability: {baseline_prob:.1%}")
print(f"Recommended strategy predicted success probability: {recommended_prob:.1%}")
print(f"Estimated uplift: {(recommended_prob - baseline_prob):.1%} (relative: {(recommended_prob/baseline_prob - 1):.0%})")

Recommended category: ENTERTAINMENT
Baseline (typical app) predicted success probability: 45.5%
Recommended strategy predicted success probability: 75.0%
Estimated uplift: 29.5% (relative: 65%)


In [9]:
rng = np.random.default_rng(42)
n = len(X_test)

# Precompute both probability arrays ONCE (this was the slow part —
# your old code called predict_proba() 500 times; this calls it just twice)
baseline_probs_all = rf.predict_proba(baseline_scenario)[:, 1]
recommended_probs_all = rf.predict_proba(recommended_scenario)[:, 1]

uplift_samples = []
for _ in range(200):   # 200 is plenty for a stable 95% CI here
    idx = rng.integers(0, n, n)
    b = baseline_probs_all[idx].mean()
    r = recommended_probs_all[idx].mean()
    uplift_samples.append(r - b)

ci_low, ci_high = np.percentile(uplift_samples, [2.5, 97.5])
print(f"Estimated uplift: {np.mean(uplift_samples):.1%}")
print(f"95% CI: [{ci_low:.1%}, {ci_high:.1%}]")

Estimated uplift: 29.5%
95% CI: [28.5%, 30.5%]


In [10]:
# Example: does a "free" listing improve store-page conversion vs a paid listing?
# Baseline conversion rate and target (informed by industry benchmarks or a pilot)
baseline_rate = 0.12    # e.g. 12% of store-page visitors install
target_rate = 0.14      # the lift we want to detect

effect = proportion_effectsize(target_rate, baseline_rate)
n_per_group = NormalIndPower().solve_power(effect_size=effect, alpha=0.05, power=0.80)

print(f"Effect size (Cohen's h): {effect:.3f}")
print(f"Required sample size per group: {n_per_group:.0f}")
print(f"Total visitors needed: {2 * n_per_group:.0f}")



Effect size (Cohen's h): 0.060
Required sample size per group: 4432
Total visitors needed: 8865


## Business Recommendation

**Decision:** Launch a free app in the **Entertainment** category.

**Specification:**
- Price: Free (Free apps succeed at 37.9% vs 0-4.8% for any paid tier — Step 5, p < 0.001)
- Size: Target the 75th percentile app size in-category (larger apps succeed more — ρ=0.31, Step 5/7)
- Update cadence: At least every 30 days (56.7% vs 18.5% success rate for stale apps — Step 4/5)
- Content Rating: Everyone (negligible standalone effect — Step 7)

**Why Entertainment:**
- Success rate: 83.7% (vs. 35.2% dataset average) — highest of any major category — Step 4
- Top-10 concentration: 72.1% (moderately concentrated, but success rate dominates the opportunity score) — Step 4
- Opportunity score: 47.7, highest of all categories with 50+ apps — Step 8
- Chi-square test confirms category significantly affects success (p = 3.53e-161, Cramér's V = 0.299) — Step 5

**Estimated impact:**
- Predicted success probability: baseline 45.5% → recommended strategy 75.0%
- Estimated uplift: **+29.5 percentage points** (95% CI: [28.5%, 30.5%]), a 65% relative increase
- Model: Random Forest, ROC-AUC 0.828, validated with 5-fold CV (0.819 ± 0.004) — Step 7

**How we would verify this in production:**
- Run a Google Play Console store-listing A/B test comparing the recommended strategy against a standard/unoptimized listing
- Required sample: **4,432 visitors per arm (8,865 total)** to detect a 2-point conversion lift (12%→14%) at 80% power, α=0.05
- Guardrail metric: rating must not drop below 4.0; stop the test early if it does
- Track Day-7 retention and install-to-review rate as secondary success signals

**Risks and limitations:**
- Dataset is a 2018 snapshot; true install counts are unknown (only bucket floors, e.g. "1,000,000+")
- Findings show association, not proven causation — Entertainment's high success rate may reflect broad user demand more than any single app-level choice
- The 45.5% baseline reflects the test set's category mix, not a uniform average across all categories; most of the uplift comes from category selection itself, not just size/price/recency choices
- Model excludes marketing spend, brand reputation, and ASO — all real drivers of success not captured in this dataset

In [11]:
summary = """# Google Play Store Growth Analytics — Executive Summary

**Question:** Which category, pricing, and app design choices maximize the odds of
reaching 1M+ installs for a new Android app launch?

**Approach:** Analyzed 9,658 unique apps using SQL, statistical hypothesis testing,
and machine learning.

**Key findings:**
1. Category matters enormously — success rates range from 4.6% (Medical) to 83.7%
   (Entertainment), an 18x gap (p < 0.001).
2. Free apps succeed far more than paid apps (37.9% vs 0-4.8%, p < 0.001).
3. Apps updated within 30 days succeed 3x more often than apps untouched for a year.
4. Larger apps succeed more often, reversing the initial hypothesis that smaller
   apps install faster.
5. A Random Forest model using only pre-launch factors predicts success with
   ROC-AUC 0.828, confirming category, size, and update recency as the top drivers.

**Recommendation:** Launch a free app in Entertainment, sized near the 75th
percentile for the category, updated at least monthly.

**Estimated impact:** Predicted success probability rises from 45.5% to 75.0%
(95% CI: [28.5%, 30.5%] uplift) versus a typical, unoptimized launch.

**Next step:** Validate with a Google Play Console store-listing A/B test
(4,432 visitors/arm, 8,865 total, 80% power) before committing to a full launch.
"""

with open("../reports/executive_summary.md", "w") as f:
    f.write(summary)

print("Saved.")

Saved.
